In [1]:
# Import required libraries
import torch
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Import our modules
import data
import model
from training import train_model

In [2]:
model_weights_filepath = r"weights/model_4_sensors_1000Hz_epoch065.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_instance = model.Attention1DConv()

state = torch.load(
    f=model_weights_filepath,
    map_location=device,
    weights_only=False)

model_instance.load_state_dict(state['model_state_dict'])
model_instance.to(device)

Attention1DConv(
  (body): Sequential(
    (0): Conv1d(4, 32, kernel_size=(32,), stride=(1,), padding=(16,))
    (1): ReLU(inplace=True)
    (2): TimeAttentionModule(
      (to_time): Conv1d(32, 1, kernel_size=(1,), stride=(1,))
      (temporal): Sequential(
        (0): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,), groups=32)
        (1): ReLU(inplace=True)
      )
      (gate): Sigmoid()
    )
    (3): ChannelAttentionModule(
      (avg_pool): AdaptiveAvgPool1d(output_size=1)
      (conv1): Conv1d(32, 16, kernel_size=(1,), stride=(1,))
      (conv2): Conv1d(16, 32, kernel_size=(1,), stride=(1,))
      (relu): ReLU(inplace=True)
      (sigmoid): Sigmoid()
    )
    (4): Conv1d(32, 32, kernel_size=(16,), stride=(2,), padding=(8,))
    (5): ReLU(inplace=True)
    (6): TimeAttentionModule(
      (to_time): Conv1d(32, 1, kernel_size=(1,), stride=(1,))
      (temporal): Sequential(
        (0): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(1,), groups=32)
        (1)

In [3]:
ds = data.ImpactDataset(
    data_dir="dataset/1000Hz/",
    force_bounds=(2, 3)
    )
loader = data.ImpactDataloader(dataset=ds)
test_loader = loader["test"]

In [4]:
test_loader

In [5]:
# make predictions with test_loader and log results

predictions = {
    "loc_preds" : [],  # list of location predicitons of the model
    "loc_labels": [], # list of location labels
    "force_preds" : [], # list of force class predictions
    "force_labels" : [] # list of force labels
}

with torch.inference_mode():
    for batch in tqdm(test_loader):
        X = batch[0].to(device)
        xy, force_logits = model_instance(X)
        predictions["loc_preds"].append(xy.detach().cpu())
        predictions["force_preds"].append(torch.argmax(torch.softmax(force_logits, dim=1), dim=1).detach().cpu())

        predictions["loc_labels"].append(batch[1].detach().cpu())
        predictions["force_labels"].append(batch[-1].detach().cpu())


  0%|          | 0/34 [00:00<?, ?it/s]

In [6]:
# Concatenate all predictions and labels
loc_preds = torch.cat(predictions["loc_preds"], dim=0)  # Shape: (N, 2)
loc_labels = torch.cat(predictions["loc_labels"], dim=0)  # Shape: (N, 2)
force_preds = torch.cat(predictions["force_preds"], dim=0)  # Shape: (N,)
force_labels = torch.cat(predictions["force_labels"], dim=0)  # Shape: (N,)

# Calculate location error (Euclidean distance)
location_errors = torch.sqrt(torch.sum((loc_preds - loc_labels) ** 2, dim=1))

# Calculate statistics for location error
mean_error = location_errors.mean().item()
percentile_95 = torch.quantile(location_errors, 0.95).item()
percentile_99 = torch.quantile(location_errors, 0.99).item()

# Calculate force classification accuracy
force_accuracy = (force_preds == force_labels).float().mean().item()

# Print results
print("=" * 50)
print("Location Error Metrics (normalized and board units):")
print("=" * 50)
print(f"Mean Error:   {mean_error:10.4f} {mean_error * 80:6.2f} mm")
print(f"95th %ile:    {percentile_95:10.4f} {percentile_95 * 80:6.2f} mm")
print(f"99th %ile:    {percentile_99:10.4f} {percentile_99 * 80:6.2f} mm")
print()
print("=" * 50)
print("Force Classification Metrics:")
print("=" * 50)
print(f"Accuracy: {force_accuracy * 100:.2f}%")
print(f"Correct:  {(force_preds == force_labels).sum().item()} / {len(force_labels)}")
print("=" * 50)

Location Error Metrics (normalized and board units):
Mean Error:       0.0445   3.56 mm
95th %ile:        0.0899   7.20 mm
99th %ile:        0.1571  12.57 mm

Force Classification Metrics:
Accuracy: 99.08%
Correct:  1078 / 1088
